## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker

### To be used by employees of Insurellm, an Insurance Tech company

### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors


### PART A: Divide our documents into chunks


Why divide documents into chunks?

The documents have different bits of information on them, like hr records, performance - when a user asks a question, it is likely about one specific part of the document. If we built one vector for one document, it is much less likely that a question is going to match directly with an entire document.

Hence split document -> best chance that one particular fragment will match with questions. Just the right about of granularity.

This part, chunking, is full of trial and error, there's no 'magical formula'.


In [5]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import (
    OpenAIEmbeddings,
)  # langchain has multiple different packages (can pip install or add it via uv), langchain_openai is the packages pertaining to OpenAi
from langchain_chroma import Chroma
from langchain_huggingface import (
    HuggingFaceEmbeddings,
)  # langchain_huggingface is the packages pertaining to HuggingFace
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
)  # langchain_community is like the 'community contributions' equivalent of ed donner's course, documunt_loaders is to load in open source contributions
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
)  # this is responsible for taking documents and turn it into chunks

# for LiteLLM -> 1 package only and can use any LLM, Langchain has many packages - more heavyweight and more learning curve

from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [ ]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(
    knowledge_base_path, recursive=True
)  # recursive = True tells glob.glob(), when you see ** in the path pattern, search through all nested folders too.
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [7]:
# How many tokens in all the documents? (we also did this week 1)

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

# even though technically we could fit all of this tokens in 1 message to OpenAI, it would be costly. The idea of RAG is to support massive databases and still allow for high quality answers when making a call to the LLM

Total tokens for gpt-4.1-nano: 63,555


In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")  # get all the folders

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    print(doc_type)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},  # make sure it works on pcs and macs
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

products
contracts
company
employees
Loaded 76 documents


In [ ]:
documents[1]

Document(metadata={'source': 'knowledge-base/products/Claimllm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Claimllm\n\n## Summary\n\nClaimllm is Insurellm's revolutionary claims processing platform that transforms the claims experience for insurers, adjusters, and policyholders. Powered by advanced AI, machine learning, and computer vision, Claimllm automates claims handling across all insurance lines—from first notice of loss through final settlement. By dramatically reducing processing time, improving accuracy, and enhancing fraud detection, Claimllm enables insurers to deliver exceptional claims service while significantly reducing operational costs. The platform seamlessly integrates with existing policy administration and core systems to create a unified insurance ecosystem.\n\n## Features\n\n### 1. Intelligent FNOL Processing\nClaimllm's AI-powered first notice of loss intake captures claim details through multiple channels including mobile apps, web portal

In [13]:
documents[1].metadata

{'source': 'knowledge-base/products/Claimllm.md', 'doc_type': 'products'}

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

# There are many text splitter classes in langchain, the RecursiveCharacterTextSplitter just does it such that it first tries to split where there's multiple empty lines between sections, then by end of sentences etc. as there's a hierarchy
# CharacterTextSplitter is the simpler one, literally splits it by character.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
)  # why have an overlap -> if we split the chunks by 1000 characters, and no overlap, sometimes the answer might span between 2 chunks, like sort of in the middle, then our answer is incomplete. By ensuring there's some overlap, we make sure that the answer from LLM is most complete.

# Try different chunk sizes and overlaps and see what works best for you
# how to know what works best? see later days in week 5.
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.' metadata={'source': 'knowledge-base/products/Rellm.md', '

In [16]:
print(chunks[100])

page_content='7. **Business Continuity:** Insurellm provides disaster recovery with 4-hour RTO (Recovery Time Objective) and 1-hour RPO (Recovery Point Objective).

---

## Renewal

This agreement includes a mutual 120-day renewal notice period. National Claims Network receives guaranteed enterprise pricing for renewal equal to or better than new enterprise customers at renewal time. Contract may be extended in 12-month increments with mutual written agreement.

---

## Features

National Claims Network will receive the complete Claimllm Enterprise suite:

1. **Unlimited Claims Processing:** No volume restrictions, supporting National's processing of 100,000+ claims annually with scalability to 500,000+ claims as business grows.

2. **White-Label Platform:** Complete branding customization including:
   - Custom domain names (claims.nationalclaimsnetwork.com)
   - Branded mobile apps (iOS and Android)
   - Customized email templates and communications
   - Co-branded claimant portals' 

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).


In [39]:
# Pick an embedding model

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(
        persist_directory=db_name, embedding_function=embeddings
    ).delete_collection()  # if the vector DB exists, wipe the db

vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=db_name
)  # Chroma is a langchain object that represents the chroma DB (open sourced - with sqlite working behind)
# embedding is where we pass in the embedding model
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 413 documents


In [40]:
# Let's investigate the vectors

collection = vectorstore._collection
# a collection is like a table in relational databases.
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
# 384 dimensions means 384 numbers in the vector. (384 dimensions is the feature of the huggingface embedding model, not the chroma DB, chroma can take in vectors of any sizes.)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 3,072 dimensions in the vector store


### Part C: Visualize!


In [41]:
# Prework

result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])  # pull out vectors from our db
documents = result["documents"]  # pull out documents (the chunks)
metadatas = result["metadatas"]  # also pull of the metadata
doc_types = [
    metadata["doc_type"] for metadata in metadatas
]  # get the doc_type from the metadata only
colors = [
    ["blue", "green", "red", "orange"][
        ["products", "employees", "contracts", "company"].index(t)
    ]
    for t in doc_types
]
# have a colour that we will pick based on its doc type

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)
# t-SNE is good at taking higher dimension data and projecting it to 2 dimensions. such that things that are close in 2D are likely to be close together in 3D and other multi-dimensional space.

tsne = TSNE(
    n_components=2, random_state=42
)  # the random_state means it should come up with the same answer each time.
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)
# there's no meaning to the x and y axis, the data is just distributed there, what matters is how close the data is to each other.

# vectors are stored by chroma, but vectors are created by the huggingface embedding model (INTERESTING! WE ONLY pass in the words to the embedding model and did not specify what document type it is and somehow the vectors have be created and grouped quite neatly, employees vectors are close and contracts vectors are also in another cluster.)

# from our testing, the OpenAI model chunks the documents better.

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(
    n_components=3, random_state=42
)  # number of components = 3 means 3 dimensions.
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()